# Langgraph Hierarchical Process

In [1]:
import operator
from typing import Annotated, Sequence, TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END, START

In [2]:
# ============================================================================
# 1. SETUP LLM & TOOLS
# ============================================================================

# Menggunakan model Qwen 0.6B sesuai environment Anda
llm = ChatOllama(
    model="qwen3:0.6b-q4_K_M",
    temperature=0.5, # Turunkan temperature agar lebih patuh
    base_url="http://localhost:11434"
)

# --- Definisi Tools ---

@tool("search_tool")
def search_tool(query: str) -> str:
    """Useful to search for information on the internet."""
    # Simulasi hasil search
    return (f"Hasil pencarian '{query}': AI Agents adalah sistem otonom yang menggunakan LLM "
            f"sebagai otak. LangGraph adalah library untuk membangun stateful, multi-agent applications "
            f"dengan kontrol siklus (loops) yang presisi.")

@tool("calculator_tool")
def calculator_tool(expression: str) -> str:
    """Useful for making calculations."""
    try:
        return f"Hasil: {eval(expression)}"
    except:
        return "Error"

@tool("write_file_tool")
def write_file_tool(filename: str, content: str) -> str:
    """Useful to write a report to a file."""
    return f"Berhasil menulis ke file '{filename}'."

tools = [search_tool, calculator_tool, write_file_tool]

In [3]:
# ============================================================================
# 2. HELPER & STATE
# ============================================================================

def create_agent(llm, tools, system_prompt):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
    ])
    if tools:
        llm_with_tools = llm.bind_tools(tools)
    else:
        llm_with_tools = llm
    return prompt | llm_with_tools

class AgentState(TypedDict):
    # operator.add penting agar pesan tidak saling menimpa, tapi bertambah (append)
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next: str
    task_completed: bool  # Flag untuk tracking apakah task sudah selesai

In [4]:
# ============================================================================
# 3. DEFINE NODES (WORKERS) - Sesuai dengan CrewAI Template
# ============================================================================

def researcher_node(state):
    print("\n--- [RESEARCHER] Sedang Bekerja ---")
    # System prompt sesuai dengan role di CrewAI
    system_prompt = """Anda adalah Researcher yang berpengalaman.
    Goal: Mencari dan mengumpulkan informasi yang relevan tentang topik yang diminta.
    Backstory: Anda adalah seorang peneliti berpengalaman yang ahli dalam 
    mencari informasi akurat dan relevan. Anda selalu memberikan data yang 
    terverifikasi dan dapat dipercaya.
    
    Fokus pada pencarian informasi tentang AI Agents dan LangGraph."""
    
    agent = create_agent(llm, [search_tool], system_prompt)
    result = agent.invoke(state)
    return {"messages": [AIMessage(content=result.content, name="Researcher")]}

def analyst_node(state):
    print("\n--- [ANALYST] Sedang Menganalisis ---")
    # System prompt sesuai dengan role di CrewAI
    system_prompt = """Anda adalah Data Analyst yang handal.
    Goal: Menganalisis informasi yang dikumpulkan dan memberikan insight yang berguna.
    Backstory: Anda adalah seorang analis data yang handal. Anda mampu 
    mengidentifikasi pola, tren, dan memberikan kesimpulan yang actionable 
    dari data yang ada.
    
    Analisis informasi dari Researcher tentang AI Agents dan LangGraph."""
    
    agent = create_agent(llm, [calculator_tool], system_prompt)
    result = agent.invoke(state)
    return {"messages": [AIMessage(content=result.content, name="Analyst")]}

def writer_node(state):
    print("\n--- [WRITER] Sedang Menulis Laporan ---")
    # System prompt sesuai dengan role di CrewAI
    system_prompt = """Anda adalah Technical Writer profesional.
    Goal: Menulis laporan yang jelas, terstruktur, dan mudah dipahami.
    Backstory: Anda adalah seorang technical writer profesional yang 
    mampu mengubah informasi kompleks menjadi dokumen yang mudah dipahami 
    oleh berbagai audience.
    
    Buat laporan komprehensif dalam format markdown tentang AI Agents dan LangGraph."""
    
    agent = create_agent(llm, [write_file_tool], system_prompt)
    result = agent.invoke(state)
    return {"messages": [AIMessage(content=result.content, name="Writer")], "task_completed": True}

In [5]:
# ============================================================================
# 4. DEFINE SUPERVISOR (MANAGER) - Sesuai dengan Manager di CrewAI
# ============================================================================

def supervisor_node(state):
    print("\n--- [SUPERVISOR] Memutuskan Langkah Selanjutnya ---")
    
    # Cek apakah task sudah completed
    if state.get("task_completed", False):
        print("   -> Task sudah selesai, finishing...")
        return {"next": "FINISH"}
    
    # Role Supervisor: Project Manager yang mengkoordinasikan tim
    # Tim terdiri dari: Researcher, Analyst, Writer
    # Tugas: Koordinasikan tim untuk menyelesaikan riset tentang AI Agents dan LangGraph
    
    # Ambil pesan terakhir untuk context
    last_message = state["messages"][-1]
    last_sender = getattr(last_message, "name", "User")
    
    # Logika delegasi yang lebih fleksibel (mirip hierarchical manager)
    decision = "FINISH"
    
    # Logika estafet dasar (bisa dikembangkan lebih complex)
    if last_sender == "User" or last_sender not in ["Researcher", "Analyst", "Writer"]:
        # Mulai dari Researcher
        decision = "Researcher"
    elif last_sender == "Researcher":
        # Setelah research, lanjut ke analysis
        decision = "Analyst"
    elif last_sender == "Analyst":
        # Setelah analysis, lanjut ke writing
        decision = "Writer"
    elif last_sender == "Writer":
        # Setelah writing, selesai
        decision = "FINISH"
    
    print(f"   -> Last sender: {last_sender}")
    print(f"   -> Decision: {decision}")
    
    return {"next": decision}

In [6]:
# ============================================================================
# 5. BUILD GRAPH - Hierarchical Pattern
# ============================================================================

workflow = StateGraph(AgentState)

# Tambah Node
workflow.add_node("Researcher", researcher_node)
workflow.add_node("Analyst", analyst_node)
workflow.add_node("Writer", writer_node)
workflow.add_node("Supervisor", supervisor_node)

# Flow Hierarchical:
# START -> Supervisor (Manager menentukan siapa yang bekerja)
workflow.add_edge(START, "Supervisor")

# Setelah Worker selesai, lapor kembali ke Supervisor
workflow.add_edge("Researcher", "Supervisor")
workflow.add_edge("Analyst", "Supervisor")
workflow.add_edge("Writer", "Supervisor")

# Supervisor mengarahkan berdasarkan keputusan
workflow.add_conditional_edges(
    "Supervisor",
    lambda x: x["next"],
    {
        "Researcher": "Researcher",
        "Analyst": "Analyst",
        "Writer": "Writer",
        "FINISH": END
    }
)

graph = workflow.compile()

In [7]:
# ============================================================================
# 6. RUN - Task sesuai dengan CrewAI Template
# ============================================================================

# Task description yang sama dengan CrewAI
task = """
Lakukan riset mendalam tentang "AI Agents dan LangGraph" kemudian:
1. Kumpulkan informasi tentang konsep dasar, use cases, dan implementasi
2. Analisis kelebihan dan kekurangan dari teknologi ini
3. Buat laporan komprehensif dalam format markdown

Pastikan laporan mencakup:
- Penjelasan konsep
- Contoh penggunaan
- Analisis pro/cons
- Rekomendasi implementasi
"""

print("=== MEMULAI WORKFLOW HIERARCHICAL ===")
print(f"Task: {task}")
print("="*80)

inputs = {
    "messages": [HumanMessage(content=task)],
    "task_completed": False
}

# Jalankan stream dengan tracking
try:
    final_state = None
    step_count = 0
    node_sequence = []
    
    for output in graph.stream(inputs, {"recursion_limit": 15}):
        step_count += 1
        node_name = list(output.keys())[0]
        node_sequence.append(node_name)
        
        print(f"\n--- Step {step_count}: {node_name} ---")
        
        # Simpan state terakhir
        final_state = output[node_name]
        
        # Tampilkan info step
        if "next" in final_state:
            print(f"Next: {final_state['next']}")
        if "task_completed" in final_state:
            print(f"Task Completed: {final_state['task_completed']}")
        if "messages" in final_state and final_state["messages"]:
            last_msg = final_state["messages"][-1]
            sender = getattr(last_msg, "name", "Unknown")
            preview = last_msg.content[:100] + "..." if len(last_msg.content) > 100 else last_msg.content
            print(f"Message from [{sender}]: {preview}")
    
    print("\n" + "="*80)
    print("WORKFLOW SELESAI")
    print("="*80)
    
    # Summary
    print(f"\n📊 SUMMARY:")
    print(f"   Total Steps: {step_count}")
    print(f"   Node Sequence: {' → '.join(node_sequence)}")
    print(f"   Task Status: {'✅ COMPLETED' if final_state.get('task_completed', False) else '❌ NOT COMPLETED'}")
    
    # Tampilkan semua messages dalam state akhir
    if final_state and "messages" in final_state:
        print(f"\n📝 TOTAL MESSAGES: {len(final_state['messages'])}")
        print("="*80)
        
        for i, msg in enumerate(final_state["messages"], 1):
            sender = getattr(msg, "name", "System")
            msg_type = type(msg).__name__
            
            print(f"\n{i}. [{sender}] ({msg_type})")
            print("-" * 80)
            print(msg.content)
    
    # Final Result
    print("\n" + "="*80)
    print("🎯 HASIL AKHIR")
    print("="*80)
    
    # Cari pesan dari Writer (hasil akhir laporan)
    writer_messages = [msg for msg in final_state.get("messages", []) 
                      if getattr(msg, "name", "") == "Writer"]
    
    if writer_messages:
        print("\n📄 LAPORAN FINAL DARI WRITER:")
        print("-" * 80)
        print(writer_messages[-1].content)
    else:
        print("\n⚠️ Tidak ada output dari Writer")
    
    print("\n" + "="*80)
    print("END OF EXECUTION")
    print("="*80)
    
except Exception as e:
    print(f"\n❌ Terjadi error eksekusi: {e}")
    import traceback
    traceback.print_exc()

=== MEMULAI WORKFLOW HIERARCHICAL ===
Task: 
Lakukan riset mendalam tentang "AI Agents dan LangGraph" kemudian:
1. Kumpulkan informasi tentang konsep dasar, use cases, dan implementasi
2. Analisis kelebihan dan kekurangan dari teknologi ini
3. Buat laporan komprehensif dalam format markdown

Pastikan laporan mencakup:
- Penjelasan konsep
- Contoh penggunaan
- Analisis pro/cons
- Rekomendasi implementasi


--- [SUPERVISOR] Memutuskan Langkah Selanjutnya ---
   -> Last sender: None
   -> Decision: Researcher

--- Step 1: Supervisor ---
Next: Researcher

--- [RESEARCHER] Sedang Bekerja ---

--- Step 2: Researcher ---
Message from [Researcher]: 

--- [SUPERVISOR] Memutuskan Langkah Selanjutnya ---
   -> Last sender: Researcher
   -> Decision: Analyst

--- Step 3: Supervisor ---
Next: Analyst

--- [ANALYST] Sedang Menganalisis ---

--- Step 4: Analyst ---
Message from [Analyst]: 

--- [SUPERVISOR] Memutuskan Langkah Selanjutnya ---
   -> Last sender: Analyst
   -> Decision: Writer

--- Step